# causal-probe: Round 2 / B: Qwen2.5-7B behaviour check

Read `KAGGLE.md` ("Round 2") in the repository first. **What this run does:** per-item answer logits of the 7B model (4-bit; float16 over two GPUs if 4-bit loading fails) on the debiased pool, the official Corr2Cause test set, CLadder and the original probe pool; no interventions. **Estimated time:** 1.5 to 3 hours on a T4.

**On Kaggle:** settings (right-hand panel) **Accelerator = GPU T4 x2**, **Internet = On**, no Hugging Face token needed (if your `HF_TOKEN` secret is attached to this notebook it is used; otherwise the public Qwen weights are downloaded anonymously and the first cell says so). Then *Save Version → Save & Run All (Commit)*. Output: the zip files in **`/kaggle/working/kaggle_out`** (Output tab of the saved version, folder `kaggle_out`).

**On Colab:** Runtime → Change runtime type → **T4 GPU**; then Runtime → **Run all**; no Hugging Face token needed (if your `HF_TOKEN` secret is attached to this notebook it is used; otherwise the public Qwen weights are downloaded anonymously and the first cell says so). Keep the browser tab open until the last cell has finished: a free Colab session stops when the tab is closed or idle for long, and its files are deleted. A cell near the end downloads the zip files from `/content/out` through the browser (allow multiple downloads when asked).

The notebook works on either platform: the first code cell detects where it is running, writes the archives to the place that
platform keeps, and stops before any GPU time is spent if it cannot. **The last cell lists the output directory with sizes and
fails loudly if an expected archive is not there.**

Everything heavy happens in `scripts/gpu_round2.py`: each stage is time-limited, skipped if already done, and allowed to fail without
stopping the rest. Afterwards put the downloaded files into `kaggle_in/r2_qwen7b/` on the laptop (see `notes/kaggle_ingest.md`).

In [ ]:
# ---- settings ---------------------------------------------------------------
COMMIT      = "6d9ce8dbe7146b10c10e7285c276a786d3727fd2"   # the code this notebook runs (pinned)
RUN_NAME    = "r2_qwen7b"
MODELS      = "qwen7b-4bit"
JOBS        = "behaviour,export"
FALLBACK    = "qwen7b-4bit:qwen7b"
NEEDS_TOKEN = False
SMOKE_TEST  = False        # True: a few items per stage (about 10 minutes) to check the plumbing
SKIP_STAGES = ""           # comma-separated stage-name prefixes to skip, e.g. "r2.qwen3b.patch"

In [ ]:
import os, subprocess, sys, json, shutil, time
t_start = time.time()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else ""
print(gpu or "NO GPU FOUND: switch the accelerator to a GPU (Kaggle: notebook settings; Colab: Runtime -> Change runtime type). The run would be far too slow on CPU.")

# ---- which platform is this, and where must the output go? --------------------
# Kaggle keeps ONLY /kaggle/working after "Save & Run All". Colab keeps nothing after the session, so the
# archives are downloaded through the browser from /content/out. Kaggle's image also has a /content folder,
# which is why Kaggle is tested first.
def choose_dirs(env, isdir):
    """-> platform, output directory, scratch directory, the path this platform keeps (None: no rule)"""
    if env.get("KAGGLE_KERNEL_RUN_TYPE") or isdir("/kaggle"):
        return "kaggle", "/kaggle/working/kaggle_out", "/kaggle/temp", "/kaggle/working"
    if env.get("COLAB_RELEASE_TAG") or env.get("COLAB_GPU") or isdir("/content"):
        return "colab", "/content/out", "/content/tmp", "/content/out"
    return "local", os.path.abspath("kaggle_out"), os.path.abspath("kaggle_tmp"), None

def under(path, parent):
    return (os.path.abspath(path) + "/").startswith(os.path.abspath(parent) + "/")

PLATFORM, WORK, TEMP, PERSIST = choose_dirs(os.environ, os.path.isdir)
os.makedirs(TEMP, exist_ok=True); os.makedirs(WORK, exist_ok=True)
# Stop here, before any GPU time is spent, if the archives would be written where the platform does not keep them.
if PERSIST and not under(WORK, PERSIST):
    raise RuntimeError(f"output directory {WORK} is not under {PERSIST}: on {PLATFORM} the archives would be lost")
with open(WORK + "/.write_test", "w") as f:
    f.write("ok")
os.remove(WORK + "/.write_test")
print("platform:", PLATFORM, "| archives will be written to", WORK,
      {"kaggle": "(kept by Kaggle: Output tab of the saved version, folder kaggle_out)",
       "colab": "(NOT kept after the session: the download cell near the end hands them to the browser)"}.get(PLATFORM, ""))

# ---- Hugging Face token, from this platform's Secrets ---------------------------
# It is never printed or written to disk by this notebook. A secret is attached PER NOTEBOOK: on Kaggle,
# Add-ons -> Secrets -> tick HF_TOKEN for this notebook (a new or imported notebook starts with none attached).
def read_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    if PLATFORM == "kaggle":
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle Secrets"
    if PLATFORM == "colab":
        from google.colab import userdata
        return userdata.get("HF_TOKEN"), "Colab Secrets"
    return None, ""
try:
    _tok, _where = read_token()
    if not _tok:
        raise LookupError("empty")
    os.environ["HF_TOKEN"] = _tok
    del _tok
    print("HF_TOKEN loaded from", _where)
except Exception as e:
    print("No HF_TOKEN secret is attached to this notebook (%s)." % type(e).__name__)
    if NEEDS_TOKEN:
        raise RuntimeError("This run needs the HF_TOKEN secret (the Gemma weights are gated). Attach it to THIS notebook "
                           "(Kaggle: Add-ons -> Secrets -> tick HF_TOKEN; Colab: the key icon in the left bar) and run again.")
    print("This run does not need one: the Qwen weights are public and are downloaded anonymously.")

os.environ["HF_HOME"] = TEMP + "/hf"                 # model downloads (not part of the output)
os.environ["CAUSAL_PROBE_CACHE"] = TEMP + "/cache"   # full activation cache (not part of the output)
os.environ["CAUSAL_PROBE_JOBS"] = "4" if (os.cpu_count() or 2) >= 4 else "2"
os.environ["PYTHONPATH"] = TEMP + "/causal-probe/src"   # the package is used straight from the checkout

In [ ]:
# ---- code at the pinned commit ------------------------------------------------
REPO_DIR = TEMP + "/causal-probe"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/skape007-ult/causal-probe.git", REPO_DIR], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR)
subprocess.run(["git", "checkout", "--quiet", COMMIT], cwd=REPO_DIR, check=True)
print("code at", subprocess.run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR, capture_output=True, text=True).stdout)

In [ ]:
# ---- pinned environment -------------------------------------------------------
# Level 1: a fresh Python 3.12 virtual environment with the full freeze from the laptop (requirements.txt).
# Level 2: the same venv with the pinned direct dependencies only (requirements-kaggle.txt).
# Level 3: this machine's own Python and torch, plus the missing packages (versions then differ; recorded in the results).
def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print("FAILED:", cmd, "\n", (r.stdout + r.stderr)[-1500:])
    return r.returncode == 0

VENV = TEMP + "/venv"
PY, level = None, None
extra = ""
if os.path.exists(VENV + "/bin/python") and os.path.exists(TEMP + "/install_level"):
    PY, level = VENV + "/bin/python", open(TEMP + "/install_level").read()
    if level.startswith("3"): PY = sys.executable
else:
    if sh(f"{sys.executable} -m pip install -q uv") and sh(f"uv venv --python 3.12 {VENV}"):
        if sh(f"uv pip install -q --python {VENV}/bin/python -r {REPO_DIR}/requirements.txt{extra} && uv pip install -q --python {VENV}/bin/python -e {REPO_DIR}"):
            PY, level = VENV + "/bin/python", "1 (full freeze)"
        elif sh(f"uv pip install -q --python {VENV}/bin/python -r {REPO_DIR}/requirements-kaggle.txt{extra} && uv pip install -q --python {VENV}/bin/python -e {REPO_DIR}"):
            PY, level = VENV + "/bin/python", "2 (pinned direct dependencies)"
    if PY is None:
        sh(f"{sys.executable} -m pip install -q sae-lens peft accelerate{extra}")
        PY, level = sys.executable, "3 (this machine's own packages; versions differ from the laptop)"
    open(TEMP + "/install_level", "w").write(level)
print("install level:", level, "| python:", PY, "| %.0f min so far" % ((time.time() - t_start) / 60))
r = subprocess.run([PY, "-c", "import torch, transformers, sklearn, causal_probe; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())"], capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])
if "4bit" in MODELS:
    # 4-bit loading needs bitsandbytes. Installed separately and allowed to fail, so that it cannot
    # change the pinned packages above; without it the runner falls back to float16 on two GPUs.
    if PY != sys.executable:
        sh(f"uv pip install -q --no-deps --python {PY} bitsandbytes")
    else:
        sh(f"{sys.executable} -m pip install -q --no-deps bitsandbytes")
    r = subprocess.run([PY, "-c", "import torch, bitsandbytes; print('bitsandbytes', bitsandbytes.__version__, '| torch still', torch.__version__)"], capture_output=True, text=True)
    print(r.stdout or "bitsandbytes not importable (the float16 two-GPU fallback will be used): " + r.stderr[-300:])

In [ ]:
# ---- run (resumable: re-running this cell skips finished stages) ---------------
cmd = [PY, "-u", "scripts/gpu_round2.py", "--run", RUN_NAME, "--models", MODELS, "--jobs", JOBS, "--out", WORK]
if FALLBACK: cmd += ["--fallback", FALLBACK]
if SMOKE_TEST: cmd.append("--tiny")
if SKIP_STAGES: cmd += ["--skip", SKIP_STAGES]
print(" ".join(cmd))
p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print("runner exit code", p.returncode, "| total %.0f min" % ((time.time() - t_start) / 60))

In [ ]:
# ---- what ran, and which archives to expect -----------------------------------
RUN = RUN_NAME + ("_tiny" if SMOKE_TEST else "")      # a smoke test runs under its own name and results folder
st = REPO_DIR + "/results/" + ("_tiny/" if SMOKE_TEST else "") + "round2_status_" + RUN + ".json"
s = json.load(open(st)) if os.path.exists(st) else {}
for k in s.get("planned", []):
    v = s["stages"].get(k)
    print("not run         " + k if v is None else ("ok     " if v["ok"] else "FAILED ") + f"{v['minutes']:6.1f} min  {k}" + ("" if v["ok"] else "   <- " + v["detail"]))
print("export:", json.dumps(s.get("export", {}).get("zip_megabytes", {})))
not_finished = "" if s.get("finished") else "the runner did not reach its export step (no finished flag in " + st + ")"
# expected: the small results archive, every archive the runner recorded, and one activation archive per model whose cache stage succeeded
expected = ["results_" + RUN + ".zip"] + list(s.get("export", {}).get("zip_megabytes", {}))
for m in s.get("models_run", []):
    if any(s.get("stages", {}).get(f"r2.{m}.{j}", {}).get("ok") for j in ("cache", "behaviour.cache")):
        expected.append(f"cache_{RUN}_{m}.zip")
expected = list(dict.fromkeys(expected))
zips = sorted(f for f in os.listdir(WORK) if f.endswith(".zip"))

In [ ]:
# ---- Colab only: hand the archives to the browser -------------------------------
# Colab deletes /content when the session ends, so download now. If the browser blocks the downloads,
# open the Files panel (folder icon, left), go to /content/out and download from there.
if PLATFORM == "colab":
    try:
        from google.colab import files
        for f in zips:
            files.download(WORK + "/" + f)
    except Exception as e:
        print("automatic download not available here:", type(e).__name__)
else:
    print("not on Colab: nothing to do in this cell")

In [ ]:
# ---- final check: are the archives where this platform keeps them? --------------
import zipfile

def check_outputs(work, persist, expected):
    """List the output directory with sizes. Returns the list of problems (empty: all expected archives are in place)."""
    problems, total = [], 0
    print("output directory:", work)
    for root, _, fs in sorted(os.walk(work)):
        for f in sorted(fs):
            p = os.path.join(root, f); n = os.path.getsize(p); total += n
            print(f"  {n / 1e6:10.1f} MB  {os.path.relpath(p, work)}")
    print(f"  {total / 1e6:10.1f} MB  in total")
    if persist and not under(work, persist):
        problems.append(f"the output directory {work} is not under {persist}, the only place this platform keeps")
    if not expected:
        problems.append("no archive was expected: the runner left no record of what it exported")
    for f in expected:
        p = os.path.join(work, f)
        if not os.path.isfile(p):
            problems.append("expected archive is missing: " + p)
        elif not zipfile.is_zipfile(p) or os.path.getsize(p) < 1000:
            problems.append("expected archive is empty or not a zip file: " + p)
    return problems

problems = ([not_finished] if not_finished else []) + check_outputs(WORK, PERSIST, expected)
if problems:
    print("\n" + "!" * 100)
    print("OUTPUT CHECK FAILED. Do not close this session before saving what is listed above by hand.")
    for q in problems:
        print("  - " + q)
    print("!" * 100)
    raise RuntimeError("OUTPUT CHECK FAILED: " + "; ".join(problems))
print("\nOUTPUT CHECK PASSED:", len(expected), "archive(s) in", WORK)
print({"kaggle": "Download them from the Output tab of this saved version (folder kaggle_out).",
       "colab": "They were handed to the browser by the cell above; if not, download them from the Files panel now."}.get(PLATFORM, ""))